---
## 14. Real-World Practice — The Titanic Dataset

We've been building toward this all lecture: real historical data, 891 passenger records from
the RMS Titanic (1912) — the original, full passenger list, with a `Survived` column (0 = No,
1 = Yes) at the center of it. Every stage we practiced today — Read, Explore, Select, Filter,
Clean, Group, Combine, Transform — applies directly here.

**This section is yours.** The setup below loads the data — from here on, work through the task
list independently, the same way you'll be expected to work on real data going forward. We're
**not** doing full exploratory analysis yet (that's next lecture) — just applying today's
mechanics.

We'll finish with a short class discussion, so keep notes on anything that surprises you.

In [1]:
import numpy as np
import pandas as pd

In [2]:
titanic = pd.read_csv(r"D:\Local\Mix\NTI-ML\Materials\Session 3\titanic-1.csv")
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


A quick note on this version of the dataset: unlike a lot of trimmed-down Titanic samples
you'll see online, this one has the full original columns, including a genuine unique key —
`PassengerId` — plus `Name`, `Ticket`, and `Cabin` in their raw form. A few things that changes
for us:
- **Duplicates become a much more meaningful check.** With a real unique ID per row, any
  `duplicated()` result (checked across all columns) tells you something is *actually* wrong with
  the data — it's not just "these passengers happen to share every visible feature" the way it
  would be without an ID.
- **`Cabin` is mostly missing** (most passengers, especially in lower classes, have no recorded
  cabin) — a different, more extreme missing-data situation than anything in our Employees table.
- **`Name` is real, raw text** — including each passenger's title (`Mr.`, `Mrs.`, `Miss.`,
  `Master.`, and a few rarer ones) embedded right in the string. That's genuinely useful data,
  not just a label, once you pull it apart with `.str` methods.

### Your tasks

1. **Explore**: run `.shape`, `.info()`, and `.describe()`. How many passengers? Which columns
   have missing data, and how much for each? (`Age`, `Cabin`, and `Embarked` are the ones to
   watch — how do their missing-value situations differ from each other?)
2. **Select**: select just `Survived`, `Pclass`, `Sex`, `Age`, and `Fare` into a new DataFrame.
3. **Filter**: find all passengers who were female, in 1st class (`Pclass == 1`), and survived
   (`Survived == 1`).
4. **Sort**: sort passengers by `Fare`, highest first, and look at the top 10 — now that we have
   real `Name` values, who were they?
5. **Create a column**: create `AgeGroup`, bucketing `Age` into `"Child"` (< 13), `"Teen"`
   (13–19), `"Adult"` (20–59), `"Senior"` (60+) — handle missing ages sensibly (decide how, and
   note your choice).
6. **Handle missing values**: check `.isnull().sum()`. Decide and justify a strategy for `Age`,
   for `Cabin`, and for `Embarked` — all three are missing for very different reasons and at very
   different rates, so a single one-size-fits-all strategy probably isn't right here.
7. **Remove duplicates**: check `titanic.duplicated().sum()`. Given the note above about
   `PassengerId` being a real unique key, what do you expect this number to be — and what would a
   *nonzero* result actually mean if you saw one?
8. **Group and aggregate**: find the survival rate (`mean` of `Survived`) grouped by `Sex`, and
   separately by `Pclass`. What pattern do you see?
9. **Pivot table**: build a pivot table of survival rate with `Pclass` as rows and `Sex` as
   columns.
10. **Transform**: extract each passenger's **title** (`Mr.`, `Mrs.`, `Miss.`, `Master.`, ...)
    out of the `Name` column using `.str` methods (hint: every title sits between a comma and a
    period — `.str.split(",")` and `.str.split(".")`, or `.str.extract()` with a regex, both get
    you there). Once you have it, check the survival rate per title — does it tell a different
    story than `Sex` alone?

Work through these in order — each builds naturally on the last, the same way today's lecture
did.

In [3]:
# Task 1: Explore
print("Shape:", titanic.shape)
print("\nDataFrame Info:")
titanic.info()
print("\nDescriptive Statistics:")
titanic.describe()
print("\nMissing Values:")
print(titanic.isnull().sum())

Shape: (891, 12)

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Descriptive Statistics:

Missing Values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket          

In [4]:
# Task 2: Select
titanic_subset = titanic[['Survived', 'Pclass', 'Sex', 'Age', 'Fare']]
print(titanic_subset.head())

   Survived  Pclass     Sex   Age     Fare
0         0       3    male  22.0   7.2500
1         1       1  female  38.0  71.2833
2         1       3  female  26.0   7.9250
3         1       1  female  35.0  53.1000
4         0       3    male  35.0   8.0500


In [5]:
# Task 3: Filter
female_first_class_survived = titanic[(titanic['Sex'] == 'female') & 
                                       (titanic['Pclass'] == 1) & 
                                       (titanic['Survived'] == 1)]
print("Count: ", len(female_first_class_survived))
print(female_first_class_survived[['Name', 'Pclass', 'Sex', 'Survived', 'Fare']])

Count:  91
                                                  Name  Pclass     Sex  \
1    Cumings, Mrs. John Bradley (Florence Briggs Th...       1  female   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)       1  female   
11                            Bonnell, Miss. Elizabeth       1  female   
31      Spencer, Mrs. William Augustus (Marie Eugenie)       1  female   
52            Harper, Mrs. Henry Sleeper (Myna Haxtun)       1  female   
..                                                 ...     ...     ...   
856         Wick, Mrs. George Dennick (Mary Hitchcock)       1  female   
862  Swift, Mrs. Frederick Joel (Margaret Welles Ba...       1  female   
871   Beckwith, Mrs. Richard Leonard (Sallie Monypeny)       1  female   
879      Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)       1  female   
887                       Graham, Miss. Margaret Edith       1  female   

     Survived      Fare  
1           1   71.2833  
3           1   53.1000  
11          1   26.550

In [6]:
# Task 4: Sort
top_10_fares = titanic.nlargest(10, 'Fare')[['Name', 'Pclass', 'Sex', 'Fare', 'Survived']]
print("Top 10 passengers by Fare:")
print(top_10_fares)

Top 10 passengers by Fare:
                                      Name  Pclass     Sex      Fare  Survived
258                       Ward, Miss. Anna       1  female  512.3292         1
679     Cardeza, Mr. Thomas Drake Martinez       1    male  512.3292         1
737                 Lesurer, Mr. Gustave J       1    male  512.3292         1
27          Fortune, Mr. Charles Alexander       1    male  263.0000         0
88              Fortune, Miss. Mabel Helen       1  female  263.0000         1
341         Fortune, Miss. Alice Elizabeth       1  female  263.0000         1
438                      Fortune, Mr. Mark       1    male  263.0000         0
311             Ryerson, Miss. Emily Borie       1  female  262.3750         1
742  Ryerson, Miss. Susan Parker "Suzette"       1  female  262.3750         1
118               Baxter, Mr. Quigg Edmond       1    male  247.5208         0


In [7]:
# Task 5: Create AgeGroup
titanic['Age_filled'] = titanic['Age'].fillna(titanic['Age'].median())
titanic['AgeGroup'] = pd.cut(titanic['Age_filled'], 
                              bins=[0, 13, 20, 60, 100], 
                              labels=['Child', 'Teen', 'Adult', 'Senior'],
                              right=False)
print(titanic[['Age', 'Age_filled', 'AgeGroup']].head(15))
print("\nAgeGroup value counts:")
print(titanic['AgeGroup'].value_counts())

     Age  Age_filled AgeGroup
0   22.0        22.0    Adult
1   38.0        38.0    Adult
2   26.0        26.0    Adult
3   35.0        35.0    Adult
4   35.0        35.0    Adult
5    NaN        28.0    Adult
6   54.0        54.0    Adult
7    2.0         2.0    Child
8   27.0        27.0    Adult
9   14.0        14.0     Teen
10   4.0         4.0    Child
11  58.0        58.0    Adult
12  20.0        20.0    Adult
13  39.0        39.0    Adult
14  14.0        14.0     Teen

AgeGroup value counts:
AgeGroup
Adult     701
Teen       95
Child      69
Senior     26
Name: count, dtype: int64


In [8]:
# Task 6: Handle missing values
print("Missing values before handling:")
print(titanic.isnull().sum())

# Drop Cabin (77%+ missing - too sparse)
titanic_clean = titanic.drop('Cabin', axis=1)

# Fill Embarked with mode (most common port)
titanic_clean['Embarked'] = titanic_clean['Embarked'].fillna(titanic_clean['Embarked'].mode()[0])

print("\nMissing values after handling:")
print(titanic_clean.isnull().sum())

Missing values before handling:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
Age_filled       0
AgeGroup         0
dtype: int64

Missing values after handling:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Embarked         0
Age_filled       0
AgeGroup         0
dtype: int64


In [9]:
# Task 7: Remove duplicates
print(f"Number of duplicate rows: {titanic_clean.duplicated().sum()}")
# Expected: 0 (PassengerId is a real unique key)

Number of duplicate rows: 0


In [10]:
# Task 8: Group and aggregate
print("Survival rate by Sex:")
print(titanic_clean.groupby('Sex')['Survived'].mean())

print("\nSurvival rate by Pclass:")
print(titanic_clean.groupby('Pclass')['Survived'].mean())
print("\nPattern: Women and 1st class passengers had higher survival rates")

Survival rate by Sex:
Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64

Survival rate by Pclass:
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

Pattern: Women and 1st class passengers had higher survival rates


In [11]:
# Task 9: Pivot table
survival_pivot = pd.pivot_table(titanic_clean, 
                                 values='Survived', 
                                 index='Pclass', 
                                 columns='Sex', 
                                 aggfunc='mean')
print("Survival rate by Pclass and Sex:")
print(survival_pivot)

Survival rate by Pclass and Sex:
Sex       female      male
Pclass                    
1       0.968085  0.368852
2       0.921053  0.157407
3       0.500000  0.135447


In [12]:
# Task 10: Transform -- extract Title from Name
titanic_clean['Title'] = titanic_clean['Name'].str.split(', ').str[1].str.split('.').str[0]

print("Unique titles:")
print(titanic_clean['Title'].unique())

print("\nSurvival rate by Title:")
print(titanic_clean.groupby('Title')['Survived'].mean().sort_values(ascending=False))

Unique titles:
['Mr' 'Mrs' 'Miss' 'Master' 'Don' 'Rev' 'Dr' 'Mme' 'Ms' 'Major' 'Lady'
 'Sir' 'Mlle' 'Col' 'Capt' 'the Countess' 'Jonkheer']

Survival rate by Title:
Title
the Countess    1.000000
Mlle            1.000000
Sir             1.000000
Ms              1.000000
Lady            1.000000
Mme             1.000000
Mrs             0.792000
Miss            0.697802
Master          0.575000
Col             0.500000
Major           0.500000
Dr              0.428571
Mr              0.156673
Jonkheer        0.000000
Rev             0.000000
Don             0.000000
Capt            0.000000
Name: Survived, dtype: float64


### 💬 Class discussion

Once you've worked through the tasks above, be ready to discuss:
- What was the single most surprising pattern you found in survival rates?
- Which missing-value strategy did you choose for `Age`? For `Cabin`? Why might a column that's
  77%+ missing (`Cabin`) call for a completely different approach than one that's ~20% missing
  (`Age`)?
- Did `titanic.duplicated().sum()` come out to `0`? What does having a real `PassengerId` change
  about how much you can trust that result, compared to a dataset without a unique key?
- Did extracting `Title` from `Name` reveal anything survival-related that `Sex` alone didn't
  (think about `Master.` vs `Mr.`, for instance)?
- Looking back at the full workflow — Read → Explore → Select → Filter → Clean → Group → Combine
  → Transform — which single stage do you think matters most for getting *trustworthy* results?
  Why?

---
## Wrap-up

```
CSV / Excel → Read Data → Explore → Select → Filter → Clean → Group → Combine → Transform → Ready for EDA
```

Every box in that diagram is now something you can actually do. Today you practiced, on both a
small employees dataset and a real historical one:
- Creating and inspecting `Series` and `DataFrame` objects
- Reading and writing CSV and Excel files
- Exploring structure and summary statistics
- Selecting with columns, `.loc`, and `.iloc`
- Filtering with conditions, `isin()`, and `between()`
- Sorting by values and by index
- Creating, renaming, dropping, and retyping columns
- Handling missing values and duplicates, thoughtfully rather than automatically
- `groupby` and aggregation — the split-apply-combine pattern
- Combining tables with `merge()` (and every join type) and `concat()`
- Reshaping with `pivot()` and `pivot_table()`
- Transforming data with `apply()`, `map()`, `lambda`, and vectorized string methods

**Next time:** hands-on practice continues, then a Mini Project, Homework, and a Kahoot to test
understanding. After that: full Exploratory Data Analysis (EDA) — we'll take these exact
mechanics and use them to actually *investigate* a dataset, including visualization.